EE 344 Final Project - Data Preprocessing

In [2]:
from google.colab import drive
import numpy as np

# drive.mount('/content/drive') # Commented out as files are directly in notebook environment

In [3]:
import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import pandas as pd
import glob


In [5]:
import pandas as pd
import os

#------------------------------
# Load data & test train split
#------------------------------

# File paths (assuming the unzipped CSV files are directly in /content/)
file1 = "/content/2014.csv"
file2 = "/content/2015.csv"

# Columns to keep
columns_to_keep = [
    'FL_DATE', 'OP_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'DEP_DELAY'
]

# Define smaller dtypes to save memory
dtypes = {
    'OP_CARRIER': 'category',
    'ORIGIN': 'category',
    'DEST': 'category',
    'CRS_DEP_TIME': 'int32',
    'CRS_ARR_TIME': 'int32',
    'CRS_ELAPSED_TIME': 'float32',
    'DISTANCE': 'float32',
    'CANCELLED': 'int8',
    'DIVERTED': 'int8',
    'DEP_DELAY': 'float32',
    'CANCELLATION_CODE': 'category'
}

# Read CSV files with selected columns and dtypes
df1 = pd.read_csv(file1, usecols=columns_to_keep, dtype=dtypes)
df2 = pd.read_csv(file2, usecols=columns_to_keep, dtype=dtypes)

# Combine into one DataFrame
all_data = pd.concat([df1, df2], ignore_index=True)

# Free memory from intermediate DataFrames
del df1, df2

# Convert date column to datetime
all_data.loc[:, 'FL_DATE'] = pd.to_datetime(all_data['FL_DATE'])

# Remove cancelled/diverted flights to reduce size
all_data = all_data[(all_data['CANCELLED'] == 0) & (all_data['DIVERTED'] == 0)].copy()

# Sort chronologically
all_data = all_data.sort_values("FL_DATE").reset_index(drop=True)

# 70/30 chronological train/test split
split_index = int(len(all_data) * 0.7)
train_df = all_data.iloc[:split_index].copy()
test_df  = all_data.iloc[split_index:].copy()

# Check date ranges
print(train_df["FL_DATE"].min(), "to", train_df["FL_DATE"].max())
print(test_df["FL_DATE"].min(), "to", test_df["FL_DATE"].max())

2014-01-01 00:00:00 to 2015-05-29 00:00:00
2015-05-29 00:00:00 to 2015-12-31 00:00:00


In [6]:
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.preprocessing import StandardScaler

# ------------------------
# Preprocessing Pipeline
# ------------------------
# Used for all models. Depending on model needs, only certain columns will be chosen for each model's data.

def preprocess_flight_data(df):
    df = df.copy()

    # Ensure FL_DATE is datetime
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

    # Extract basic time features from CRS_DEP_TIME
    df['CRS_DEP_TIME'] = df['CRS_DEP_TIME'].fillna(0).astype(int)
    df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100  # 0-23 hour
    df['DEP_MIN'] = df['CRS_DEP_TIME'] % 100    # minutes if needed
    df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek  # Monday=0, Sunday=6
    df['MONTH'] = df['FL_DATE'].dt.month

    # Time-of-day label
    def get_time_of_day(hour):
        if 5 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        else:
            return 'night'
    df['TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

    # Holiday flag using USFederalHolidayCalendar
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['FL_DATE'].min(), end=df['FL_DATE' ].max())
    df['IS_HOLIDAY'] = df['FL_DATE'].isin(holidays).astype(int)

    # Cyclic encoding placeholders (for linear models)
    df['DEP_HOUR_SIN'] = np.sin(2 * np.pi * df['DEP_HOUR']/24)
    df['DEP_HOUR_COS'] = np.cos(2 * np.pi * df['DEP_HOUR']/24)
    df['DAY_OF_WEEK_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK']/7)
    df['DAY_OF_WEEK_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK']/7)

    return df

# Apply preprocessing to all data.
train_data = preprocess_flight_data(train_df)
test_data = preprocess_flight_data(test_df)

print(train_data.head())

     FL_DATE OP_CARRIER ORIGIN DEST  CRS_DEP_TIME  DEP_DELAY  CRS_ARR_TIME  \
0 2014-01-01         AA    ICT  DFW          1135        9.0          1300   
1 2014-01-01         UA    SAN  SFO          1122       18.0          1257   
2 2014-01-01         UA    SFO  LAX          1631       -7.0          1804   
3 2014-01-01         UA    IAH  LAX          1318       25.0          1514   
4 2014-01-01         UA    AUS  DEN           751       -4.0           906   

   CANCELLED CANCELLATION_CODE  DIVERTED  ...  DEP_HOUR  DEP_MIN  DAY_OF_WEEK  \
0          0               NaN         0  ...        11       35            2   
1          0               NaN         0  ...        11       22            2   
2          0               NaN         0  ...        16       31            2   
3          0               NaN         0  ...        13       18            2   
4          0               NaN         0  ...         7       51            2   

   MONTH  TIME_OF_DAY  IS_HOLIDAY DEP_HOUR_S

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ------------------------
# Memory-efficient prepare_model_data
# ------------------------

def prepare_model_data(df, approach='binary', model_type='linear'):
    """
    Prepares X and y for flight delay prediction.

    Linear models: uses sparse one-hot encoding + scaled numeric + cyclic features
    Tree models: uses categorical codes + numeric features
    """
    df = df.copy()  # Only one copy at start

    # -------------------------
    # Target column
    # -------------------------
    if approach == 'binary':
        y = (df['DEP_DELAY'] > 15).astype(int)
    elif approach == 'regression':
        y = df['DEP_DELAY'].astype(float)
    elif approach == 'multiclass':
        # FIX: Changed the first bin to -np.inf to include negative/zero delays in the first category
        bins = [-np.inf, 15, 30, 60, np.inf]
        labels = [0, 1, 2, 3]
        y = pd.cut(df['DEP_DELAY'], bins=bins, labels=labels, right=True).astype(int)
    else:
        raise ValueError("Approach must be 'binary', 'regression', or 'multiclass'")

    # -------------------------
    # Linear models
    # -------------------------
    if model_type == 'linear':
        # Column selection
        categorical_cols_linear = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK']
        numeric_cols_linear = ['DISTANCE', 'CRS_ELAPSED_TIME'] + ['DEP_HOUR_SIN', 'DEP_HOUR_COS', 'DAY_OF_WEEK_SIN', 'DAY_OF_WEEK_COS']

        # Sparse one-hot encoder for categorical features
        ohe = OneHotEncoder(drop='first', sparse_output=True, handle_unknown='ignore')
        X_cat_sparse = ohe.fit_transform(df[categorical_cols_linear])

        # Scale numeric features
        scaler = StandardScaler()
        X_num_scaled = scaler.fit_transform(df[numeric_cols_linear])

        # Combine numeric + categorical (sparse)
        from scipy.sparse import hstack
        X = hstack([X_num_scaled, X_cat_sparse], format='csr')

    # -------------------------
    # Tree models
    # -------------------------
    elif model_type == 'tree':
        tree_categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK', 'TIME_OF_DAY']
        tree_numeric_cols = ['DISTANCE', 'CRS_ELAPSED_TIME']

        # Convert categorical columns to category codes (memory-efficient)
        X_cat = df[tree_categorical_cols].apply(lambda x: x.astype('category').cat.codes)
        X_num = df[tree_numeric_cols]

        # Concatenate numeric + categorical codes
        X = pd.concat([X_num, X_cat], axis=1)

    else:
        raise ValueError("model_type must be 'linear' or 'tree'")

    return X, y

In [8]:
# Binary -- For example. Each approach will be put in it's own notebook due to RAM limitations, but they are shown below commented out.
# Linear model (sparse one-hot)
#X_train_linear_bin, y_train_bin = prepare_model_data(train_data, approach='binary', model_type='linear')
#X_test_linear_bin, y_test_bin = prepare_model_data(test_data, approach='binary', model_type='linear')

# Tree model (categorical codes)
#X_train_tree_bin, y_train_bin_tree = prepare_model_data(train_data, approach='binary', model_type='tree')
#X_test_tree_bin, y_test_bin_tree = prepare_model_data(test_data, approach='binary', model_type='tree')

# Regression
# Linear model
#X_train_linear_reg, y_train_reg = prepare_model_data(train_data, approach='regression', model_type='linear')
#X_test_linear_reg, y_test_reg = prepare_model_data(test_data, approach='regression', model_type='linear')

# Tree model
#X_train_tree_reg, y_train_reg_tree = prepare_model_data(train_data, approach='regression', model_type='tree')
#X_test_tree_reg, y_test_reg_tree = prepare_model_data(test_data, approach='regression', model_type='tree')

# Multiclass
# Linear model
#X_train_linear_multi, y_train_multi = prepare_model_data(train_data, approach='multiclass', model_type='linear')
#X_test_linear_multi, y_test_multi = prepare_model_data(test_data, approach='multiclass', model_type='linear')

# Tree model
#X_train_tree_multi, y_train_multi_tree = prepare_model_data(train_data, approach='multiclass', model_type='tree')
#X_test_tree_multi, y_test_multi_tree = prepare_model_data(test_data, approach='multiclass', model_type='tree')

Multi Class

In [9]:
# Multiclass target
X_train_mc, y_train_mc = prepare_model_data(train_data, approach='multiclass', model_type='tree')
X_test_mc, y_test_mc = prepare_model_data(test_data, approach='multiclass', model_type='tree')

# Check class distribution
print("Training class counts:\n", y_train_mc.value_counts())
print("Test class counts:\n", y_test_mc.value_counts())

Training class counts:
 DEP_DELAY
0    6408297
1     593368
2     495296
3     477709
Name: count, dtype: int64
Test class counts:
 DEP_DELAY
0    2828019
1     217706
3     190860
2     181131
Name: count, dtype: int64


In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight




Random Forest

In [11]:
X_train_mc, y_train_mc = prepare_model_data(train_data, approach='multiclass', model_type='tree')
X_test_mc, y_test_mc = prepare_model_data(test_data, approach='multiclass', model_type='tree')

print("Training class counts:\n", y_train_mc.value_counts())
print("Test class counts:\n", y_test_mc.value_counts())


Training class counts:
 DEP_DELAY
0    6408297
1     593368
2     495296
3     477709
Name: count, dtype: int64
Test class counts:
 DEP_DELAY
0    2828019
1     217706
3     190860
2     181131
Name: count, dtype: int64


In [12]:
# ----------------------------
# Random Forest with Oversampling
# ----------------------------
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# ----------------------------
# Stratified subset (250k train, 50k test)
# ----------------------------
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train_mc, y_train_mc,
    train_size=250_000,
    stratify=y_train_mc,
    random_state=42
)

X_test_sub, _, y_test_sub, _ = train_test_split(
    X_test_mc, y_test_mc,
    train_size=50_000,
    stratify=y_test_mc,
    random_state=42
)

# ----------------------------
# Oversample minority classes in training set
# ----------------------------
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_sub, y_train_sub)

# ----------------------------
# Random Forest
# ----------------------------
rf = RandomForestClassifier(
    n_estimators=150,     # medium number of trees
    max_depth=15,         # limit depth for generalization
    random_state=42,
    n_jobs=-1
)

# Train on oversampled data
rf.fit(X_train_ros, y_train_ros)

# Predict on original test subset
y_pred = rf.predict(X_test_sub)

# ----------------------------
# Evaluation
# ----------------------------
print("Classification Report:\n")
print(classification_report(y_test_sub, y_pred, digits=4))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test_sub, y_pred))

macro_f1 = f1_score(y_test_sub, y_pred, average='macro')
print("\nRandom Forest Macro F1-score:", macro_f1)

Classification Report:

              precision    recall  f1-score   support

           0     0.8517    0.6941    0.7649     41373
           1     0.0797    0.0628    0.0702      3185
           2     0.0669    0.0883    0.0761      2650
           3     0.0803    0.2955    0.1263      2792

    accuracy                         0.5995     50000
   macro avg     0.2696    0.2852    0.2594     50000
weighted avg     0.7179    0.5995    0.6484     50000

Confusion Matrix:

[[28715  1966  2713  7979]
 [ 1954   200   281   750]
 [ 1531   162   234   723]
 [ 1513   182   272   825]]

Random Forest Macro F1-score: 0.2593609685406903


Multinomial Logistic Baseline

In [13]:
# ----------------------------
# Multinomial Logistic Regression for Multiclass Classification
# ----------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# ----------------------------
# Subset for memory efficiency
# ----------------------------
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train_mc, y_train_mc,
    train_size=200_000,
    stratify=y_train_mc,
    random_state=42
)

X_test_sub, _, y_test_sub, _ = train_test_split(
    X_test_mc, y_test_mc,
    train_size=50_000,
    stratify=y_test_mc,
    random_state=42
)

# ----------------------------
# Multinomial Logistic Regression
# ----------------------------
logreg = LogisticRegression(
    multi_class='multinomial',
    solver='saga',        # good for large datasets
    max_iter=500,         # increase if convergence warnings occur
    class_weight='balanced', # handle class imbalance
    n_jobs=-1,
    random_state=42
)

# Train model
logreg.fit(X_train_sub, y_train_sub)

# Predict on test subset
y_pred = logreg.predict(X_test_sub)

# ----------------------------
# Evaluation
# ----------------------------
print("Classification Report:\n")
print(classification_report(y_test_sub, y_pred, digits=4))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test_sub, y_pred))

macro_f1 = f1_score(y_test_sub, y_pred, average='macro')
print("\nMultinomial Logistic Regression Macro F1-score:", macro_f1)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Classification Report:

              precision    recall  f1-score   support

           0     0.8671    0.3834    0.5317     41373
           1     0.0819    0.3372    0.1318      3185
           2     0.0576    0.0940    0.0714      2650
           3     0.0689    0.3521    0.1152      2792

    accuracy                         0.3633     50000
   macro avg     0.2689    0.2917    0.2125     50000
weighted avg     0.7296    0.3633    0.4585     50000

Confusion Matrix:

[[15861 10396  3555 11561]
 [  875  1074   309   927]
 [  762   834   249   805]
 [  794   806   209   983]]

Multinomial Logistic Regression Macro F1-score: 0.2125257359787231


Neural Network

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ---------------------------
# Prepare tensors
# ---------------------------
X_train_tensor = torch.tensor(X_train_mc.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_mc.values, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test_mc.values, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test_mc.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# ---------------------------
# Compute class weights
# ---------------------------
classes = np.unique(y_train_mc)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_mc)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# ---------------------------
# Define lightweight neural network
# ---------------------------
class FlightDelayNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=4):
        super(FlightDelayNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model = FlightDelayNN(input_dim=X_train_mc.shape[1])
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------
# Training loop
# ---------------------------
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

# ---------------------------
# Evaluation
# ---------------------------
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds)
        all_labels.append(y_batch)

y_pred_nn = torch.cat(all_preds).numpy()
y_true_nn = torch.cat(all_labels).numpy()

from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("\nNeural Network Performance:")
print(classification_report(y_true_nn, y_pred_nn, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true_nn, y_pred_nn))

macro_f1_nn = f1_score(y_true_nn, y_pred_nn, average='macro')
print("Neural Network Macro F1-score:", macro_f1_nn)

Epoch 1/10, Loss: 1.5035
Epoch 2/10, Loss: 1.3771
Epoch 3/10, Loss: 1.3737
Epoch 4/10, Loss: 1.3682
Epoch 5/10, Loss: 1.3598
Epoch 6/10, Loss: 1.3558
Epoch 7/10, Loss: 1.3548
Epoch 8/10, Loss: 1.3543
Epoch 9/10, Loss: 1.3541
Epoch 10/10, Loss: 1.3536

Neural Network Performance:
              precision    recall  f1-score   support

           0     0.8892    0.5099    0.6481   2828019
           1     0.0956    0.1069    0.1009    217706
           2     0.0872    0.1740    0.1162    181131
           3     0.0776    0.4842    0.1337    190860

    accuracy                         0.4650   3417716
   macro avg     0.2874    0.3187    0.2497   3417716
weighted avg     0.7509    0.4650    0.5564   3417716

Confusion Matrix:
 [[1441956  188739  263995  933329]
 [  71680   23268   36121   86637]
 [  53330   17473   31523   78805]
 [  54591   13922   29934   92413]]
Neural Network Macro F1-score: 0.24973948431373927
